# Phase 3: Feature Extraction

Raw EEG waveforms are hard for classifiers to work with directly. Feature extraction converts each (32, 640) EEG segment into a compact vector of meaningful numbers that capture the signal's characteristics.

We extract 3 categories of features (same as published research on SAM 40):
1) **Frequency-Domain**: Band power in Delta, Theta, Alpha, Beta, Gamma + Beta/Alpha ratio (stress indicator)
       → 6 features × 32 channels = 192 features
2) **Time-Domain**: Hjorth parameters (Activity, Mobility, Complexity) + Statistical features (mean, std, skewness, kurtosis)
       → 7 features × 32 channels = 224 features
3) **Non-Linear**: Spectral entropy, Higuchi fractal dimension
       → 2 features × 32 channels = 64 features

Total: 480 features per segment (before any feature selection)

**Prerequisites**:
- Phase 2 completed (preprocessed_segments.npz + segment_labels.csv)

In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch
from scipy.stats import skew, kurtosis
import time
import warnings
warnings.filterwarnings('ignore')
 
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

# Configuration

In [2]:

BASE_DIR = os.path.join("..", "")  # If running from Notebooks/
 
DATA_DIR = os.path.join(BASE_DIR, "Results", "phase2")  # Load from Phase 2 output
OUTPUT_DIR = os.path.join(BASE_DIR, "Results", "phase3")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Constants

In [3]:

SAMPLING_RATE = 128
N_CHANNELS = 32
 
CHANNEL_NAMES = [
    'Cz', 'Fz', 'Fp1', 'F7', 'F3', 'FC1', 'C3', 'FC5',
    'FT9', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO9', 'O1',
    'Pz', 'Oz', 'O2', 'PO10', 'P8', 'P4', 'CP2', 'CP6',
    'T8', 'FT10', 'FC6', 'C4', 'FC2', 'F4', 'F8', 'Fp2'
]
 
EEG_BANDS = {
    'Delta':  (0.5,  4),
    'Theta':  (4,    8),
    'Alpha':  (8,   13),
    'Beta':   (13,  30),
    'Gamma':  (30,  45),
}
 
TASK_COLORS = {
    'Relaxation': '#2ecc71', 'Stroop': '#e74c3c',
    'Mirror_Image': '#f39c12', 'Arithmetic': '#3498db',
}

# Feature Extraction Functions

## Frequency Domain Features

Compute power in each EEG frequency band using Welch's PSD method.

The brain produces oscillations at different frequencies, and each frequency range (band) is associated with different mental states:
- **Delta (0.5-4 Hz)**:  Deep sleep. Not very useful for our task since subjects are awake, but good to include.
- **Theta (4-8 Hz)**:    Memory, drowsiness. Can increase during cognitive effort (frontal theta).
- **Alpha (8-13 Hz)**:   Relaxed wakefulness. DECREASES during stress (alpha suppression / desynchronization).
- **Beta (13-30 Hz)**:   Active thinking, focus. INCREASES during stress and cognitive engagement.
- **Gamma (30-45 Hz)**:  High-level processing, concentration.

The Beta/Alpha ratio combines the two key changes (alpha down, beta up) into one powerful stress indicator.

In [6]:


def extract_band_powers(signal, fs=SAMPLING_RATE):
    
    """
    Parameters:
        signal: 1D array of EEG samples for one channel
        fs: sampling rate
    
    Returns:
        dict with 'Delta', 'Theta', 'Alpha', 'Beta', 'Gamma', 'Beta_Alpha_Ratio'
    """

    # Compute PSD using Welch's method
    freqs, psd = welch(signal, fs=fs, nperseg=min(256, len(signal)),
                       noverlap=128, scaling='density')
    freq_res = freqs[1] - freqs[0]
    
    powers = {}
    for band_name, (lo, hi) in EEG_BANDS.items():
        idx = np.where((freqs >= lo) & (freqs <= hi))[0]
        powers[band_name] = np.sum(psd[idx]) * freq_res
    
    # Beta/Alpha ratio (avoid division by zero)
    alpha = powers['Alpha']
    beta = powers['Beta']
    powers['Beta_Alpha_Ratio'] = beta / alpha if alpha > 1e-10 else 0.0
    
    return powers

## Time Domain Features

Compute Hjorth parameters: Activity, Mobility, Complexity.

These are simple but powerful features that describe the shape of a time-series signal:
- **Activity**:   Variance of the signal. Measures signal power. Higher activity = larger amplitude fluctuations.
- **Mobility**:   How quickly the signal changes. Computed as the ratio of std of the first derivative to std of the signal. Higher mobility = more frequent oscillations.
- **Complexity**: How much the signal resembles a pure sine wave.A pure sine has complexity = 1. More complex signals (multiple frequencies mixed) have higher complexity. Computed as mobility of the derivative / mobility of signal.

These are computationally cheap (just variance and derivatives) and work well for EEG classification.

In [7]:

def extract_hjorth_parameters(signal):

    """
    Parameters:
        signal: 1D array
    
    Returns:
        dict with 'Activity', 'Mobility', 'Complexity'
    """

    # Activity = variance of the signal
    activity = np.var(signal)
    
    # First derivative (discrete: difference between consecutive samples)
    d1 = np.diff(signal)
    # Second derivative
    d2 = np.diff(d1)
    
    # Mobility = sqrt(var(d1) / var(signal))
    var_d1 = np.var(d1)
    mobility = np.sqrt(var_d1 / activity) if activity > 1e-10 else 0.0
    
    # Complexity = mobility(d1) / mobility(signal)
    var_d2 = np.var(d2)
    mobility_d1 = np.sqrt(var_d2 / var_d1) if var_d1 > 1e-10 else 0.0
    complexity = mobility_d1 / mobility if mobility > 1e-10 else 0.0
    
    return {
        'Activity': activity,
        'Mobility': mobility,
        'Complexity': complexity,
    }


Compute basic statistical features of the signal.

- **Mean**:      Average amplitude (should be ~0 after normalization)
- **Std**:       Spread of amplitudes (signal variability)
- **Skewness**:  Asymmetry of the amplitude distribution (0 = symmetric, positive = right-skewed)
- **Kurtosis**:  "Peakedness" of the distribution (0 = Gaussian, positive = heavy tails/sharp peak)

In [8]:
def extract_statistical_features(signal):

    """
    Parameters:
        signal: 1D array
    Returns:
        dict with 'Mean', 'Std', 'Skewness', 'Kurtosis'
    """
    return {
        'Mean': np.mean(signal),
        'Std': np.std(signal),
        'Skewness': skew(signal),
        'Kurtosis': kurtosis(signal),
    }